# VeriFact-BHC Data Exploration
## Step 1: Setup and Data Loading

This notebook explores the MIMIC-III-Ext-VeriFact-BHC dataset (100 ICU patients) to understand its structure before building our baseline and RAG pipelines.

**Dataset source:** PhysioNet — MIMIC-III-Ext-VeriFact-BHC v1.0.0  
**Key files:**
- `reference_ehr/ehr_noteevents.csv.gz` — All clinical notes for 100 patients (our pipeline input)
- `reference_ehr/admissions.csv.gz` — Admission-level metadata
- `propositions/propositions.csv.gz` — Decomposed BHC propositions (contains our gold-standard BHC targets)
- `propositions/human_verdicts.csv.gz` — Clinician annotations for evaluation

In [9]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data/verifact-bhc")

In [10]:
list(DATA_DIR.iterdir())

[PosixPath('../data/verifact-bhc/Llama_3.1_LICENSE'),
 PosixPath('../data/verifact-bhc/propositions'),
 PosixPath('../data/verifact-bhc/.DS_Store'),
 PosixPath('../data/verifact-bhc/reference_ehr'),
 PosixPath('../data/verifact-bhc/README.md'),
 PosixPath('../data/verifact-bhc/prompts'),
 PosixPath('../data/verifact-bhc/LICENSE.txt'),
 PosixPath('../data/verifact-bhc/SHA256SUMS.txt')]

### 1.1 Load the Clinical Notes (Reference EHR)

The `ehr_noteevents.csv.gz` file contains all clinical notes for our 100 patients, pulled from the MIMIC-III `NOTEEVENTS` table. This excludes the final discharge summary (which contains the BHC we're trying to generate). These notes are what both our baseline and RAG pipeline will use as input.

In [11]:
# Load the clinical notes from the reference EHR
# These are all notes for 100 patients EXCEPT the final discharge summary
notes = pd.read_csv(DATA_DIR / "reference_ehr" / "ehr_noteevents.csv.gz")

# Quick overview: how many rows (notes) and columns do we have?
print(f"Total notes: {notes.shape[0]}")
print(f"Columns: {notes.shape[1]}")
print(f"\nColumn names:")
print(notes.columns.tolist())

Total notes: 4787
Columns: 11

Column names:
['ROW_ID', 'SUBJECT_ID', 'HADM_ID', 'CHARTDATE', 'CHARTTIME', 'STORETIME', 'CATEGORY', 'DESCRIPTION', 'CGID', 'ISERROR', 'TEXT']


### 1.2 Explore the Dataset Structure

Let's understand how notes are distributed across patients and what types of clinical notes we're working with. This matters because the diversity of note types is what makes RAG valuable — retrieving from fragmented, multi-source documentation.

In [12]:
# How many unique patients and admissions?
print(f"Unique patients (SUBJECT_ID): {notes['SUBJECT_ID'].nunique()}")
print(f"Unique admissions (HADM_ID): {notes['HADM_ID'].nunique()}")

Unique patients (SUBJECT_ID): 100
Unique admissions (HADM_ID): 125


In [13]:
# How many notes per patient? 
# This tells us how fragmented the input is for each patient
notes_per_patient = notes.groupby('SUBJECT_ID').size()

print(f"Notes per patient:")
print(f"  Min:    {notes_per_patient.min()}")
print(f"  Max:    {notes_per_patient.max()}")
print(f"  Mean:   {notes_per_patient.mean():.1f}")
print(f"  Median: {notes_per_patient.median():.1f}")

Notes per patient:
  Min:    10
  Max:    314
  Mean:   47.9
  Median: 25.0


In [14]:
# What types of clinical notes do we have?
# This shows the diversity of note categories — the "fragmentation" that RAG needs to handle
print("Note categories and counts:")
print(notes['CATEGORY'].value_counts().to_string())

Note categories and counts:
CATEGORY
Nursing              1228
Physician            1223
Radiology            1134
Respiratory           471
ECG                   306
Nutrition              92
General                91
Rehab Services         81
Echo                   71
Social Work            37
Discharge summary      26
Case Management        14
Nursing/other          12
Consult                 1


### 1.3 Note Length Analysis

Understanding the length of clinical notes helps us plan our chunking strategy for the RAG pipeline and determine whether inputs will fit within LLM context windows for the baseline.

In [15]:
# Calculate the token-approximate length of each note
# A rough estimate: 1 token ≈ 4 characters for English clinical text
notes['char_length'] = notes['TEXT'].str.len()
notes['approx_tokens'] = notes['char_length'] // 4

# Overall note length stats
print("Note length (approximate tokens):")
print(f"  Min:    {notes['approx_tokens'].min()}")
print(f"  Max:    {notes['approx_tokens'].max()}")
print(f"  Mean:   {notes['approx_tokens'].mean():.0f}")
print(f"  Median: {notes['approx_tokens'].median():.0f}")

Note length (approximate tokens):
  Min:    13
  Max:    5371
  Mean:   773
  Median: 440


In [16]:
# Total tokens per patient — this is what the baseline would need to fit in one prompt
tokens_per_patient = notes.groupby('SUBJECT_ID')['approx_tokens'].sum()

print("Total approximate tokens per patient (all notes combined):")
print(f"  Min:    {tokens_per_patient.min()}")
print(f"  Max:    {tokens_per_patient.max()}")
print(f"  Mean:   {tokens_per_patient.mean():.0f}")
print(f"  Median: {tokens_per_patient.median():.0f}")
print(f"\nPatients exceeding 128K tokens: {(tokens_per_patient > 128000).sum()}")
print(f"Patients exceeding 32K tokens:  {(tokens_per_patient > 32000).sum()}")
print(f"Patients exceeding 8K tokens:   {(tokens_per_patient > 8000).sum()}")

Total approximate tokens per patient (all notes combined):
  Min:    5720
  Max:    232852
  Mean:   36983
  Median: 17562

Patients exceeding 128K tokens: 9
Patients exceeding 32K tokens:  31
Patients exceeding 8K tokens:   91


In [17]:
# Average note length by category
# This helps us understand which note types are long-form vs. short
print("Average approximate tokens by note category:")
print(
    notes.groupby('CATEGORY')['approx_tokens']
    .agg(['mean', 'median', 'count'])
    .sort_values('mean', ascending=False)
    .round(0)
    .to_string()
)

Average approximate tokens by note category:
                     mean  median  count
CATEGORY                                
Discharge summary  3063.0  2894.0     26
Physician          1778.0  1709.0   1223
Consult            1051.0  1051.0      1
Rehab Services      810.0   861.0     81
Echo                608.0   599.0     71
Nutrition           564.0   612.0     92
Social Work         535.0   397.0     37
Nursing             486.0   432.0   1228
Radiology           401.0   304.0   1134
General             344.0   178.0     91
Respiratory         334.0   338.0    471
Nursing/other       223.0   173.0     12
Case Management     209.0   190.0     14
ECG                  55.0    51.0    306


## Step 2: Explore the BHC Targets (Gold Standard)

The `propositions.csv.gz` file contains the Brief Hospital Course narratives — both human-written (our gold standard) and LLM-generated — decomposed into propositions. We need to understand what the target BHC looks like: how long they are, and what we're asking our pipeline to produce.

In [18]:
# Load the propositions file
propositions = pd.read_csv(DATA_DIR / "propositions" / "propositions.csv.gz")

print(f"Total propositions: {propositions.shape[0]}")
print(f"Columns: {propositions.columns.tolist()}")

Total propositions: 13070
Columns: ['proposition_id', 'text', 'author_type', 'proposition_type', 'parent_text_chunk', 'brief_hospital_course', 'subject_id', 'row_id', 'hadm_id', 'admitdate', 'admittime', 'dischargedate', 'dischargetime']


In [19]:
# What are the different author types and proposition types?
print("Author types:")
print(propositions['author_type'].value_counts().to_string())
print(f"\nProposition types:")
print(propositions['proposition_type'].value_counts().to_string())

Author types:
author_type
human    8625
llm      4445

Proposition types:
proposition_type
claim       9095
sentence    3975


In [20]:
# Extract the unique human-written BHCs — these are our gold standard targets
# Each patient should have one human-written BHC
human_bhcs = (
    propositions[propositions['author_type'] == 'human']
    [['subject_id', 'hadm_id', 'brief_hospital_course']]
    .drop_duplicates(subset=['subject_id'])
)

print(f"Unique human-written BHCs: {human_bhcs.shape[0]}")

Unique human-written BHCs: 100


In [21]:
# How long are the gold standard BHCs we're trying to generate?
human_bhcs['bhc_char_length'] = human_bhcs['brief_hospital_course'].str.len()
human_bhcs['bhc_approx_tokens'] = human_bhcs['bhc_char_length'] // 4

print("Human-written BHC length (approximate tokens):")
print(f"  Min:    {human_bhcs['bhc_approx_tokens'].min()}")
print(f"  Max:    {human_bhcs['bhc_approx_tokens'].max()}")
print(f"  Mean:   {human_bhcs['bhc_approx_tokens'].mean():.0f}")
print(f"  Median: {human_bhcs['bhc_approx_tokens'].median():.0f}")

Human-written BHC length (approximate tokens):
  Min:    107
  Max:    1905
  Mean:   607
  Median: 513


In [22]:
# Let's look at one example BHC to understand what we're generating
# Pick a patient near the median length
median_length = human_bhcs['bhc_approx_tokens'].median()
example = human_bhcs.iloc[
    (human_bhcs['bhc_approx_tokens'] - median_length).abs().argsort().iloc[0]
]

print(f"Example BHC (subject_id: {example['subject_id']}, ~{example['bhc_approx_tokens']} tokens):")
print("=" * 80)
print(example['brief_hospital_course'][:2000])  # First 2000 characters
print("..." if len(example['brief_hospital_course']) > 2000 else "")

Example BHC (subject_id: 1084, ~511 tokens):
Pt is a 61 y.o male with h.o prostate ca with fistulous and
infectious complications, pelvis osteo, PE, nephrolithiasis, HL,
chronic back and LLE pain, [**Doctor Last Name 933**], UTIs, multiple leg
debridements with presents with AMS.

# AMS- secondary to medication error and exacerbated by opiate
withdrawal.  The patient was found altered at home, given narcan
and woke up to become very combative and difficult to control.
He was treated with several medications in the ED, but needed to
be intubated to have a CT scan and LP.  His CT and LP were both
negative.  He was monitored overnight and treated with zosyn for
a supicious UA.  In the AM of his second day of hospitalization,
he woke up and self-extubated himself despite being on a
propofol gtt.  His mental status began to clear over the course
of the day, and he remember that instead of taking his 2.5 tabs
of methadone, he took 2.5 tabs of ambien.  That was likely the
cause of his altered

## Step 3: Explore the Propositions and Human Annotations

Each BHC has been decomposed into propositions (sentences and atomic claims) that clinicians annotated as Supported, Not Supported, or Not Addressed by the patient's EHR. This gives us proposition-level evaluation — far more granular than just ROUGE or BERTScore against the full text.

In [23]:
# How many propositions per patient, split by author and type?
prop_summary = (
    propositions
    .groupby(['author_type', 'proposition_type'])
    .agg(
        total_props=('proposition_id', 'count'),
        unique_patients=('subject_id', 'nunique'),
        mean_per_patient=('proposition_id', lambda x: len(x) / propositions['subject_id'].nunique())
    )
    .round(1)
)
print("Propositions by author type and proposition type:")
print(prop_summary.to_string())

Propositions by author type and proposition type:
                              total_props  unique_patients  mean_per_patient
author_type proposition_type                                                
human       claim                    5648              100              56.5
            sentence                 2977              100              29.8
llm         claim                    3447              100              34.5
            sentence                  998              100              10.0


In [24]:
# Load the human annotations (clinician verdicts)
verdicts = pd.read_csv(DATA_DIR / "propositions" / "human_verdicts.csv.gz")

print(f"Total verdict rows: {verdicts.shape[0]}")
print(f"Columns: {verdicts.columns.tolist()}")

Total verdict rows: 13070
Columns: ['proposition_id', 'text', 'author_type', 'proposition_type', 'rater1', 'rater2', 'rater3', 'verdict1', 'verdict2', 'verdict3', 'uncertain_flag1', 'uncertain_flag2', 'uncertain_flag3', 'comment1', 'comment2', 'comment3', 'round1_num_raters_agree', 'round1_majority_vote', 'rater4', 'rater5', 'verdict4', 'verdict5', 'comment4', 'comment5', 'round2_decision', 'adjudicated_verdict', 'adjudicated_comment', 'human_gt']


In [25]:
# What are the ground truth labels and their distribution?
print("Ground truth label distribution (all propositions):")
print(verdicts['human_gt'].value_counts().to_string())
print(f"\nAs percentages:")
print(verdicts['human_gt'].value_counts(normalize=True).mul(100).round(1).to_string())

Ground truth label distribution (all propositions):
human_gt
Supported        9617
Not Addressed    2297
Not Supported    1156

As percentages:
human_gt
Supported        73.6
Not Addressed    17.6
Not Supported     8.8


In [26]:
# Break this down by author type — how do human-written vs LLM-written BHCs compare?
# This tells us: does the human clinician include more unsupported info than the LLM?
gt_by_author = (
    verdicts
    .groupby(['author_type', 'human_gt'])
    .size()
    .unstack(fill_value=0)
)

# Convert to percentages within each author type
gt_by_author_pct = gt_by_author.div(gt_by_author.sum(axis=1), axis=0).mul(100).round(1)

print("Ground truth labels by author type (%):")
print(gt_by_author_pct.to_string())

Ground truth labels by author type (%):
human_gt     Not Addressed  Not Supported  Supported
author_type                                         
human                 25.6           11.8       62.6
llm                    1.9            3.1       95.0


### Key Takeaway for Our Project

The human annotations give us three things:
1. **Gold standard BHC text** — what we evaluate our generated summaries against using ROUGE-L and BERTScore
2. **Proposition-level ground truth** — we can decompose our generated BHC into propositions and check what % are Supported vs Not Supported by the EHR
3. **A baseline for comparison** — we can see how our pipeline's factual accuracy compares to both the human-written and existing LLM-written BHCs

## Step 4: Dataset Summary and Pipeline Implications
Quick reference

In [27]:
# Build a concise summary of everything we've explored
print("=" * 70)
print("VERIFACT-BHC DATASET SUMMARY")
print("=" * 70)

print(f"\n📋 PATIENTS & ADMISSIONS")
print(f"   Patients: {notes['SUBJECT_ID'].nunique()}")
print(f"   Admissions: {notes['HADM_ID'].nunique()}")

print(f"\n📝 CLINICAL NOTES (Pipeline Input)")
print(f"   Total notes: {notes.shape[0]}")
print(f"   Note categories: {notes['CATEGORY'].nunique()}")
print(f"   Notes per patient: median {notes_per_patient.median():.0f}, "
      f"range {notes_per_patient.min()}-{notes_per_patient.max()}")
print(f"   Tokens per patient: median {tokens_per_patient.median():.0f}, "
      f"range {tokens_per_patient.min()}-{tokens_per_patient.max()}")
print(f"   Patients exceeding 32K tokens: {(tokens_per_patient > 32000).sum()}/100")

print(f"\n🎯 BHC TARGETS (Gold Standard)")
print(f"   Human-written BHCs: {human_bhcs.shape[0]}")
print(f"   BHC length: median {human_bhcs['bhc_approx_tokens'].median():.0f} tokens, "
      f"range {human_bhcs['bhc_approx_tokens'].min()}-{human_bhcs['bhc_approx_tokens'].max()}")

print(f"\n✅ PROPOSITION ANNOTATIONS")
print(f"   Total propositions: {propositions.shape[0]}")
print(f"   Human BHC propositions: {(propositions['author_type'] == 'human').sum()}")
print(f"   Ground truth: {verdicts['human_gt'].value_counts().to_dict()}")

print(f"\n⚠️  PIPELINE DESIGN IMPLICATIONS")
print(f"   - 31/100 patients exceed 32K tokens → baseline will hit context limits")
print(f"   - 14 note categories → RAG must handle diverse clinical documentation")
print(f"   - Median 25 notes per patient → genuine multi-document fragmentation")
print(f"   - Human BHCs contain 25.6% 'Not Addressed' info → expect moderate")
print(f"     ROUGE/BERTScore since our pipeline can only use available EHR data")
print("=" * 70)

VERIFACT-BHC DATASET SUMMARY

📋 PATIENTS & ADMISSIONS
   Patients: 100
   Admissions: 125

📝 CLINICAL NOTES (Pipeline Input)
   Total notes: 4787
   Note categories: 14
   Notes per patient: median 25, range 10-314
   Tokens per patient: median 17562, range 5720-232852
   Patients exceeding 32K tokens: 31/100

🎯 BHC TARGETS (Gold Standard)
   Human-written BHCs: 100
   BHC length: median 513 tokens, range 107-1905

✅ PROPOSITION ANNOTATIONS
   Total propositions: 13070
   Human BHC propositions: 8625
   Ground truth: {'Supported': 9617, 'Not Addressed': 2297, 'Not Supported': 1156}

⚠️  PIPELINE DESIGN IMPLICATIONS
   - 31/100 patients exceed 32K tokens → baseline will hit context limits
   - 14 note categories → RAG must handle diverse clinical documentation
   - Median 25 notes per patient → genuine multi-document fragmentation
   - Human BHCs contain 25.6% 'Not Addressed' info → expect moderate
     ROUGE/BERTScore since our pipeline can only use available EHR data
